# Diagnostic: What's Limiting Validation Score?
Test: (1) Original vs engineered features, (2) Model complexity, (3) Ridge baseline

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
import gc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load data (same as solution)
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

# Remove constant columns
const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

# Align columns
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()

print(f"Original features (no engineering): {X_train.shape}")

In [ ]:
# TEST 1: ORIGINAL FEATURES ONLY (445 features, no engineering)
print("\n" + "="*70)
print("TEST 1: XGBoost with ORIGINAL features only (445 features)")
print("="*70)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
xgb_params_orig = {
    'objective': 'reg:squarederror', 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bytree': 0.7, 'lambda': 1.0, 'alpha': 0.5,
    'random_state': 42, 'n_jobs': 4, 'verbosity': 0
}

scores_orig = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    model = xgb.XGBRegressor(**xgb_params_orig, n_estimators=800)
    model.fit(X_tr, y_tr)
    r2 = r2_score(y_val, model.predict(X_val))
    scores_orig.append(r2)
    print(f"  Fold {fold} R²: {r2:.6f}")
    del model
    gc.collect()

print(f"\n✓ XGBoost (Original) Avg R²: {np.mean(scores_orig):.6f} (std: {np.std(scores_orig):.6f})")

In [ ]:
# Add engineered features
def engineer_features(X):
    X_feat = X.copy()
    X_feat['row_mean'] = X.mean(axis=1)
    X_feat['row_std'] = X.std(axis=1)
    X_feat['row_median'] = X.median(axis=1)
    X_feat['row_skew'] = X.skew(axis=1)
    X_feat['row_max'] = X.max(axis=1)
    X_feat['row_min'] = X.min(axis=1)
    X_feat['row_max_min_diff'] = X_feat['row_max'] - X_feat['row_min']
    lag_cols = [c for c in X.columns if 'LagT' in c]
    if len(lag_cols) > 0:
        lag_subset = X[lag_cols]
        X_feat['lag_mean'] = lag_subset.mean(axis=1)
        X_feat['lag_std'] = lag_subset.std(axis=1)
    return X_feat

X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)
print(f"With engineered features: {X_train_eng.shape}")

# TEST 2: CURRENT APPROACH (454 features with engineering)
print("\n" + "="*70)
print("TEST 2: XGBoost with ENGINEERED features (454 features)")
print("="*70)

scores_eng = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_eng), 1):
    X_tr, X_val = X_train_eng.iloc[train_idx], X_train_eng.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    model = xgb.XGBRegressor(**xgb_params_orig, n_estimators=800)
    model.fit(X_tr, y_tr)
    r2 = r2_score(y_val, model.predict(X_val))
    scores_eng.append(r2)
    print(f"  Fold {fold} R²: {r2:.6f}")
    del model
    gc.collect()

print(f"\n✓ XGBoost (Engineered) Avg R²: {np.mean(scores_eng):.6f} (std: {np.std(scores_eng):.6f})")

In [ ]:
# TEST 3: SIMPLIFIED XGBoost (reduce complexity)
print("\n" + "="*70)
print("TEST 3: SIMPLER XGBoost (max_depth=3, n_estimators=200, higher regularization)")
print("="*70)

xgb_params_simple = {
    'objective': 'reg:squarederror', 'max_depth': 3, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bytree': 0.7, 'lambda': 5.0, 'alpha': 2.0,
    'random_state': 42, 'n_jobs': 4, 'verbosity': 0
}

scores_simple = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_eng), 1):
    X_tr, X_val = X_train_eng.iloc[train_idx], X_train_eng.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    model = xgb.XGBRegressor(**xgb_params_simple, n_estimators=200)
    model.fit(X_tr, y_tr)
    r2 = r2_score(y_val, model.predict(X_val))
    scores_simple.append(r2)
    print(f"  Fold {fold} R²: {r2:.6f}")
    del model
    gc.collect()

print(f"\n✓ Simpler XGBoost Avg R²: {np.mean(scores_simple):.6f} (std: {np.std(scores_simple):.6f})")

In [ ]:
# TEST 4: RIDGE REGRESSION BASELINE (linear model, simple)
print("\n" + "="*70)
print("TEST 4: Ridge Regression (linear baseline, high alpha=10)")
print("="*70)

scores_ridge = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_eng), 1):
    X_tr, X_val = X_train_eng.iloc[train_idx], X_train_eng.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    model = Ridge(alpha=10.0)
    model.fit(X_tr, y_tr)
    r2 = r2_score(y_val, model.predict(X_val))
    scores_ridge.append(r2)
    print(f"  Fold {fold} R²: {r2:.6f}")
    del model
    gc.collect()

print(f"\n✓ Ridge Regression Avg R²: {np.mean(scores_ridge):.6f} (std: {np.std(scores_ridge):.6f})")

In [ ]:
# SUMMARY COMPARISON
print("\n" + "="*70)
print("DIAGNOSTIC SUMMARY - What's Limiting Validation?")
print("="*70)

results = {
    "Original Features (445)": np.mean(scores_orig),
    "With Engineered Features (454)": np.mean(scores_eng),
    "Simpler XGBoost (depth=3, reg higher)": np.mean(scores_simple),
    "Ridge Regression Baseline": np.mean(scores_ridge)
}

for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:.<45} R² = {score:.6f}")

print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)

best_approach = max(results, key=results.get)
best_score = results[best_approach]
improvement = best_score - np.mean(scores_eng)  # vs current approach

print(f"\n✓ BEST APPROACH: {best_approach}")
print(f"  Improvement vs current: {improvement:+.6f}")

if improvement <= 0.01:
    print(f"\n⚠ ANALYSIS: All approaches show similar weak signals (~0.21 R²).")
    print(f"  Problem is LIKELY NOT model complexity or engineering.")
    print(f"  Problem is LIKELY weak feature-target correlation or data quality.")
    print(f"\n  RECOMMENDATIONS:")
    print(f"    1. Check feature correlations with TARGET")
    print(f"    2. Analyze train/test distribution match")
    print(f"    3. Consider this problem may have limited signal")
else:
    print(f"\n✓ FOUND: {best_approach} improves by {improvement:.6f}")
    print(f"  This approach should be used for submission")